In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    "font.size": 17,
    "axes.titlesize": 17,
    "axes.labelsize": 17,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17,
    "legend.fontsize": 17,
    "legend.title_fontsize": 17,
    "figure.dpi": 300,
})

In [ ]:
# =========================================================
# Consistent feature colors for panels A and B
# =========================================================
FEATURE_COLORS = {
    "Accession": "#03045e",
    "Domain Structure": "#023e8a",
    "Order": "#0077b6",
    "Class": "#0096c7",
    "cluster": "#00b4d8",
    "Functional Cluster": "#48cae4",
    "Primary Effector": "#82d0de",
}

NULL_COLOR = "#8F8F8F"
PERM_COLOR = "#F09E4C"

# =========================================================
# Better UMAP palette
# =========================================================
UMAP_PALETTE = [
    "#3B5BA5", "#E68613", "#4FA64F", "#D84B4B", "#8A63B8",
    "#7A5C58", "#D77FB3", "#6D6D6D", "#A4AD3A", "#2B9AA0",
    "#9ED4E6", "#FF9EAE", "#9C6B4E", "#AFAFAF", "#63B35D",
    "#D4B82A", "#6CB6B3", "#B18BBF", "#FFBE6F", "#7ECF8A"
]

In [ ]:
def _ensure_xy_numeric(df):
    df = df.copy()
    for c in ["x", "y"]:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.replace(",", ".", regex=False)
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

In [ ]:
def knn_label_consistency_scores(df, label_column, k=10, n_perm=200, dropna=True, seed=0):
    rng = np.random.default_rng(seed)
    df2 = _ensure_xy_numeric(df)

    cols = ["x", "y", label_column]
    if dropna:
        df2 = df2.dropna(subset=cols)

    coords = df2[["x", "y"]].to_numpy()
    labels = df2[label_column].astype("category")
    y = labels.cat.codes.to_numpy()

    knn = NearestNeighbors(n_neighbors=k).fit(coords)
    neigh_idx = knn.kneighbors(return_distance=False)

    real = np.mean([(y[neigh] == y[i]).mean() for i, neigh in enumerate(neigh_idx)])

    null_scores = []
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        s = np.mean([(y_perm[neigh] == y_perm[i]).mean() for i, neigh in enumerate(neigh_idx)])
        null_scores.append(s)

    null_mean = float(np.mean(null_scores))
    std = float(np.std(null_scores, ddof=1)) if len(null_scores) > 1 else np.nan
    return real, null_mean, std

In [ ]:
def _collapse_top_categories(
    s: pd.Series,
    top_n: int = 10,
    other_label: str = "Other",
    min_keep_count: int = 2,
    na_label: str = "NA",
):
    s2 = s.astype(str).fillna(na_label)
    vc = s2.value_counts(dropna=False)
    keep = vc[vc >= min_keep_count].head(top_n).index.tolist()

    if len(keep) == 0:
        return pd.Categorical([other_label] * len(s2), categories=[other_label], ordered=False)

    s_collapsed = s2.where(s2.isin(keep), other_label)
    categories = keep + ([other_label] if (s_collapsed == other_label).any() else [])
    return pd.Categorical(s_collapsed, categories=categories, ordered=False)

In [ ]:
def _style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.16, linewidth=0.7)
    ax.tick_params(length=0)

In [ ]:
# =========================================================
# Panel A
# =========================================================
def plot_knn_summary_panel(
    ax,
    df,
    columns,
    k=16,
    n_perm=200,
    seed=1,
):
    rows = []
    for col in columns:
        real, null_mean, std = knn_label_consistency_scores(df, col, k=k, n_perm=n_perm, seed=seed)
        rows.append({
            "column": col,
            "knn_real": real,
            "knn_null_mean": null_mean,
            "std": std
        })

    res = pd.DataFrame(rows).sort_values("knn_real", ascending=True)

    y_pos = np.arange(len(res))

    for i, (_, row) in enumerate(res.iterrows()):
        ax.barh(
            y_pos[i],
            row["knn_real"],
            color=FEATURE_COLORS.get(row["column"], "#4C78A8"),
            alpha=0.55,
            edgecolor=FEATURE_COLORS.get(row["column"], "#4C78A8"),
            linewidth=1.2,
            hatch="///",
            height=0.72,
            zorder=2,
        )

        y_pos = np.arange(len(res)) * 1.0

        # plot permutation mean with horizontal errorbar showing std
        if not np.isnan(row["std"]):
            ax.errorbar(
                row["knn_null_mean"],
                y_pos[i],
                xerr=row["std"],
                fmt='|',
                color=PERM_COLOR,
                ecolor=PERM_COLOR,
                elinewidth=2.2,
                capsize=6,
                markersize=40,
                zorder=5,
            )
        else:
            ax.plot(
                row["knn_null_mean"],
                y_pos[i],
                marker='|',
                color=PERM_COLOR,
                markersize=40,
                markeredgewidth=2.2,
                zorder=5,
            )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(res["column"])
    ax.set_xlabel(f"kNN agreement (k={k})")
    ax.set_xlim(0, max(0.95, res["knn_real"].max() + 0.05))

    _style_axis(ax)
    ax.grid(True, axis="x", alpha=0.20)
    ax.grid(False, axis="y")

    handles = [
        Line2D([0], [0], linestyle="None", marker="|", color=PERM_COLOR,
           markersize=20, markeredgewidth=2.2, label="Permutation mean"),
        plt.Rectangle(
            (0, 0), 1, 1,
            facecolor="#4C78A8",
            edgecolor="#4C78A8",
            alpha=0.55,
            hatch="///",
            linewidth=1.2,
            label="kNN agreement",
        )
    ]
    
    ax.legend(
        handles=handles,
        loc="lower right",
        frameon=False,
        handlelength=1.8,
        borderpad=0.2,
        labelspacing=0.4,
    )

    return res

In [ ]:
# =========================================================
# Panel B
# =========================================================
def plot_knn_vs_k_panel(
    ax,
    df,
    cols_main,
    ks=(1, 2, 4, 8, 16, 32, 64, 128),
    n_perm=200,
    seed=42,
):
    for idx, col in enumerate(cols_main):
        real_scores = []
        null_means = []

        for k in ks:
            real, null_mean, _ = knn_label_consistency_scores(df, col, k=k, n_perm=n_perm, seed=seed)
            real_scores.append(real)
            null_means.append(null_mean)

        color = FEATURE_COLORS.get(col, "#4C78A8")

        ax.plot(
            ks,
            real_scores,
            marker="o",
            linewidth=2.9,
            markersize=10,
            color=color,
            label=col,
        )

        null_label = "null mean (all annotations)" if idx == 0 else "_nolegend_"
        ax.plot(
            ks,
            null_means,
            linestyle="--",
            linewidth=2.4,
            color=NULL_COLOR,
            alpha=0.75,
            label=null_label,
        )

    ax.set_xlabel("k")
    ax.set_ylabel("kNN agreement")
    ax.set_xscale("log", base=2)
    ax.set_xticks(ks)
    ax.set_xticklabels([str(k) for k in ks])

    _style_axis(ax)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.1),
        ncol=4,
        frameon=False,
        handlelength=2.0,
        columnspacing=1.0,
        labelspacing=0.45,
        borderaxespad=0.0,
    )


In [ ]:
# =========================================================
# UMAP panels with separate legend axis
# =========================================================
def plot_umap_annotation_panel_with_legend(
    ax_scatter,
    ax_legend,
    df,
    col,
    x_col="x",
    y_col="y",
    point_size=18,
    alpha=0.96,
    top_n=10,
    other_label="Other",
    min_keep_count=2,
):
    df2 = _ensure_xy_numeric(df).dropna(subset=[x_col, y_col]).copy()
    x = df2[x_col].to_numpy()
    y = df2[y_col].to_numpy()

    if col not in df2.columns:
        ax_scatter.axis("off")
        ax_legend.axis("off")
        return

    s = df2[col]
    unique_n = s.nunique(dropna=False)

    if unique_n > (top_n + 1):
        s_plot = _collapse_top_categories(
            s,
            top_n=top_n,
            other_label=other_label,
            min_keep_count=min_keep_count,
        )
        labels = list(s_plot.categories)
    else:
        s2 = s.astype(str).fillna("NA")
        vc = s2.value_counts(dropna=False)
        labels = vc.index.tolist()
        s_plot = pd.Categorical(s2, categories=labels, ordered=False)

    if other_label in labels:
        labels = [lab for lab in labels if lab != other_label] + [other_label]
        s_plot = pd.Categorical(pd.Series(s_plot), categories=labels, ordered=False)

    palette = UMAP_PALETTE[:len(labels)]
    cmap = ListedColormap(palette)
    norm = BoundaryNorm(np.arange(-0.5, len(labels) + 0.5, 1), len(labels))

    code_map = {lab: i for i, lab in enumerate(labels)}
    codes = pd.Series(s_plot).map(code_map).to_numpy()

    ax_scatter.scatter(
        x, y,
        c=codes,
        cmap=cmap,
        norm=norm,
        s=point_size,
        alpha=alpha,
        linewidths=0,
        rasterized=True,
    )

    shown_top = len(labels) - 1 if other_label in labels else len(labels)
    ax_scatter.set_title(f"{col} (only top {shown_top} shown)", pad=16)
    ax_scatter.set_xlabel("UMAP-1")
    ax_scatter.set_ylabel("UMAP-2")
    _style_axis(ax_scatter)

    handles = []
    for i, lab in enumerate(labels):
        handles.append(
            Line2D(
                [0], [0],
                marker="o",
                color="none",
                markerfacecolor=palette[i],
                markeredgecolor="none",
                markersize=6.2,
                label=lab,
                alpha=alpha,
            )
        )

    ax_legend.axis("off")
    ax_legend.legend(
        handles=handles,
        title=col,
        loc="upper left",
        frameon=False,
        borderaxespad=0.0,
        handletextpad=0.5,
        labelspacing=0.35,
        columnspacing=0.7,
    )


In [ ]:
# =========================================================
# Master figure
# =========================================================
def make_lov_multi_panel_figure_publication(
    df,
    summary_cols=None,
    robustness_cols=None,
    umap_cols=None,
    k_summary=16,
    ks=(1, 2, 4, 8, 16, 32, 64, 128),
    n_perm=200,
    seed=42,
    figsize=(16, 15.5),
    output_prefix="lov_multi_panel_publication",
):
    if summary_cols is None:
        summary_cols = [
            "Accession",
            "Domain Structure",
            "Order",
            "Class",
            "cluster",
            "Functional Cluster",
            "Primary Effector",
        ]

    if robustness_cols is None:
        robustness_cols = [
            "Accession",
            "Domain Structure",
            "Order",
            "Class",
            "cluster",
            "Functional Cluster",
            "Primary Effector",
        ]

    if umap_cols is None:
        umap_cols = ["Primary Effector", "Functional Cluster", "Order", "Accession"]

    fig = plt.figure(figsize=figsize)

    # Main outer grid
    outer = fig.add_gridspec(
        3, 2,
        height_ratios=[1.75, 1.6, 1.6],
        hspace=0.4,
        wspace=0.14
    )

    # Top row
    axA = fig.add_subplot(outer[0, 0])
    axB = fig.add_subplot(outer[0, 1])

    plot_knn_summary_panel(axA, df, summary_cols, k=k_summary, n_perm=n_perm, seed=seed)
    plot_knn_vs_k_panel(axB, df, robustness_cols, ks=ks, n_perm=n_perm, seed=seed)

    # Middle-left (C)
    gsC = outer[1, 0].subgridspec(1, 2, width_ratios=[5.5, 1.70], wspace=0.06)
    axC = fig.add_subplot(gsC[0, 0])
    axC_leg = fig.add_subplot(gsC[0, 1])

    # Middle-right (D)
    gsD = outer[1, 1].subgridspec(1, 2, width_ratios=[5.5, 1.70], wspace=0.06)
    axD = fig.add_subplot(gsD[0, 0])
    axD_leg = fig.add_subplot(gsD[0, 1])

    # Bottom-left (E)
    gsE = outer[2, 0].subgridspec(1, 2, width_ratios=[5.5, 1.70], wspace=0.06)
    axE = fig.add_subplot(gsE[0, 0])
    axE_leg = fig.add_subplot(gsE[0, 1])

    # Bottom-right (F)
    gsF = outer[2, 1].subgridspec(1, 2, width_ratios=[5.5, 1.70], wspace=0.06)
    axF = fig.add_subplot(gsF[0, 0])
    axF_leg = fig.add_subplot(gsF[0, 1])

    plot_umap_annotation_panel_with_legend(
        axC, axC_leg, df, umap_cols[0], point_size=54, alpha=0.6, top_n=10
    )
    plot_umap_annotation_panel_with_legend(
        axD, axD_leg, df, umap_cols[1], point_size=54, alpha=0.6, top_n=10
    )
    plot_umap_annotation_panel_with_legend(
        axE, axE_leg, df, umap_cols[2], point_size=54, alpha=0.6, top_n=10
    )
    plot_umap_annotation_panel_with_legend(
        axF, axF_leg, df, umap_cols[3], point_size=54, alpha=0.6, top_n=10
    )

    # panel letters without bold
    letter_style = dict(fontsize=22, fontweight="normal", va="top", ha="left")
    for label, ax in zip(
        ["A", "B", "C", "D", "E", "F"],
        [axA, axB, axC, axD, axE, axF]
    ):
        ax.text(-0.10, 1.04, label, transform=ax.transAxes, **letter_style)

    plt.subplots_adjust(
        left=0.075,
        right=0.985,
        top=0.975,
        bottom=0.075
    )

    fig.savefig(f"{output_prefix}.png", dpi=600, bbox_inches="tight")
    fig.savefig(f"{output_prefix}.pdf", bbox_inches="tight")
    fig.savefig(f"{output_prefix}.svg", bbox_inches="tight")
    plt.show()




In [ ]:
df = pd.read_csv("lov_10_15.tsv", sep="\t")
print(df.shape)
df.head()

In [ ]:
summary_cols = [
    "Accession",
    "Domain Structure",
    "Order",
    "Class",
    "cluster",
    "Functional Cluster",
    "Primary Effector",
]

robustness_cols = [
    "Accession",
    "Domain Structure",
    "Order",
    "Class",
    "cluster",
    "Functional Cluster",
    "Primary Effector",
]

umap_cols = [
    "Primary Effector",
    "Functional Cluster",
    "Order",
    "Accession",
]

In [ ]:
make_lov_multi_panel_figure_publication(
    df=df,
    summary_cols=summary_cols,
    robustness_cols=robustness_cols,
    umap_cols=umap_cols,
    k_summary=16,
    ks=(1, 2, 4, 8, 16, 32, 64, 128),
    n_perm=200,
    seed=42,
    figsize=(23.5, 22.5),
    output_prefix="lov_multi_panel_publication",
)